# Frost Bronze to Silver Observations

Temperatures below -60C or above +50C are quarantined as `IMPLAUSIBLE_TEMPERATURE`. 

Frost `quality_code` values are preserved for inspection.


In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from silver_layer.frost_observations import ingest

/home/roland/UiS/rewrite/group14/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and Spark session

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "silver_layer").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"

spark = (
    SparkSession.builder.appName("Frost Silver")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
print("Project:", PROJECT_ROOT)

26/11/19 18:43:19 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/19 18:43:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 18:43:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/19 18:43:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Project: /home/roland/UiS/rewrite/group14


## Ingest
This is a full two-year run from 2024-01-01 inclusive to 2026-01-01 exclusive.


In [3]:
result = ingest(
    bronze_root=BRONZE_ROOT,
    silver_root=SILVER_ROOT,
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)
result

Checking Frost JSON files...
Checked 24,422 files; 24,422 valid, 0 rejected


Running Spark validation and writing Silver outputs...


26/11/19 18:51:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:51:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:51:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:51:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:52:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:52:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:52:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:52:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Silver rows: 76,715,902
Quarantine rows: 446,439


{'dataset': 'frost_observations',
 'status': 'success',
 'rows': 76715902,
 'quarantined_rows': 446439,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/frost/observations',
 'quarantine_path': '/home/roland/UiS/rewrite/group14/data/silver/quarantine/frost_observations'}

## Read Silver and quarantine

In [4]:
silver = spark.read.parquet(result["silver_path"])
quarantine = spark.read.parquet(result["quarantine_path"])

silver.printSchema()
print("Silver rows:", silver.count())
print("Quarantine rows:", quarantine.count())

root
 |-- station_id: string (nullable = true)
 |-- source_id: string (nullable = true)
 |-- reference_time: timestamp (nullable = true)
 |-- observation_time: timestamp (nullable = true)
 |-- temperature_c: decimal(18,6) (nullable = true)
 |-- quality_code: integer (nullable = true)
 |-- time_resolution: string (nullable = true)
 |-- time_offset: string (nullable = true)
 |-- time_series_id: integer (nullable = true)
 |-- height_m: decimal(18,6) (nullable = true)

Silver rows: 76715902
Quarantine rows: 446439


## Sample observations
The same reference time may legitimately have different offsets or resolutions.

In [5]:
silver.orderBy("station_id", "reference_time").show(20, truncate=False)

+----------+---------+-------------------+-------------------+-------------+------------+---------------+-----------+--------------+--------+
|station_id|source_id|reference_time     |observation_time   |temperature_c|quality_code|time_resolution|time_offset|time_series_id|height_m|
+----------+---------+-------------------+-------------------+-------------+------------+---------------+-----------+--------------+--------+
|SN10055   |SN10055:0|2024-12-16 00:00:00|2024-12-16 00:00:00|-0.800000    |0           |PT10M          |PT0H       |0             |2.000000|
|SN10055   |SN10055:0|2024-12-16 00:00:00|2024-12-16 00:00:00|-0.800000    |0           |PT1H           |PT0H       |0             |2.000000|
|SN10055   |SN10055:0|2024-12-16 00:10:00|2024-12-16 00:10:00|-0.800000    |0           |PT10M          |PT0H       |0             |2.000000|
|SN10055   |SN10055:0|2024-12-16 00:20:00|2024-12-16 00:20:00|-0.700000    |0           |PT10M          |PT0H       |0             |2.000000|
|SN100

## Silver quality checks
All accepted Silver temperatures must be within −60C to +50C.


In [ ]:
silver.groupBy("time_resolution").count().orderBy(F.desc("count")).show(truncate=False)

silver.select(
    F.min("reference_time").alias("first_utc"),
    F.max("reference_time").alias("last_utc"),
    F.min("temperature_c").alias("min_c"),
    F.max("temperature_c").alias("max_c"),
).show(truncate=False)

out_of_range = silver.where(
    (F.col("temperature_c") < -60) | (F.col("temperature_c") > 50)
).count()
print(f"Silver temperatures outside [-60, 50] °C: {out_of_range:,} (expected: 0)")

+---------------+--------+
|time_resolution|count   |
+---------------+--------+
|PT10M          |57366217|
|PT1H           |9218213 |
|PT1M           |5945694 |
|PT6H           |2192227 |
|PT30M          |1967235 |
|PT3H           |26316   |
+---------------+--------+



+-------------------+-------------------+----------+---------+
|first_utc          |last_utc           |min_c     |max_c    |
+-------------------+-------------------+----------+---------+
|2024-01-01 00:00:00|2025-12-31 23:59:00|-59.900000|50.000000|
+-------------------+-------------------+----------+---------+

Silver temperatures outside [-60, 50] °C: 0 (expected: 0)


In [7]:
key = [
    "source_id",
    "reference_time",
    "time_offset",
    "time_resolution",
    "time_series_id",
    "height_m",
]
print(
    "Repeated identity keys:", silver.groupBy(*key).count().where("count > 1").count()
)
print(
    "Null timestamps:",
    silver.where("reference_time IS NULL OR observation_time IS NULL").count(),
)

26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 18:56:39 WARN RowBasedKeyValueBatch: Calling spill() on

Repeated identity keys: 0


Null timestamps: 0


## Quarantine and Frost quality codes

`INVALID_OBSERVATION` means required metadata or a value failed structural validation. 

`IMPLAUSIBLE_TEMPERATURE` means an otherwise valid observation was outside our decided range. Invalid observations take precedence if both rules apply.

Frost quality codes are retained, not filtered by their numerical value.

In [8]:
print("Quarantine reasons:")
quarantine.groupBy("quarantine_reason").count().orderBy(F.desc("count")).show(
    truncate=False
)

print("Frost quality codes in accepted Silver observations:")
silver.groupBy("quality_code").count().orderBy("quality_code").show(50, truncate=False)

print("Frost quality codes in quarantined observations:")
quarantine.groupBy("quarantine_reason", "quality_code").count().orderBy(
    "quarantine_reason", "quality_code"
).show(100, truncate=False)

print("Example implausible temperatures (if any):")
quarantine.where(F.col("quarantine_reason") == "IMPLAUSIBLE_TEMPERATURE").select(
    "station_id", "reference_time", "temperature_c", "quality_code", "time_resolution"
).orderBy(F.abs(F.col("temperature_c")).desc()).show(20, truncate=False)

print("Example structurally invalid observations:")
quarantine.where(F.col("quarantine_reason") == "INVALID_OBSERVATION").select(
    "station_id",
    "source_id",
    "reference_time",
    "temperature_c",
    "quality_code",
    "element_id",
    "unit",
    "level_type",
    "level_unit",
    "_source_file",
).show(20, truncate=False)

Quarantine reasons:
+-----------------------+------+
|quarantine_reason      |count |
+-----------------------+------+
|INVALID_OBSERVATION    |445709|
|IMPLAUSIBLE_TEMPERATURE|730   |
+-----------------------+------+

Frost quality codes in accepted Silver observations:
+------------+--------+
|quality_code|count   |
+------------+--------+
|0           |68928437|
|1           |80229   |
|2           |5571642 |
|4           |250673  |
|5           |449935  |
|6           |1434986 |
+------------+--------+

Frost quality codes in quarantined observations:
+-----------------------+------------+------+
|quarantine_reason      |quality_code|count |
+-----------------------+------------+------+
|IMPLAUSIBLE_TEMPERATURE|1           |189   |
|IMPLAUSIBLE_TEMPERATURE|2           |373   |
|IMPLAUSIBLE_TEMPERATURE|6           |168   |
|INVALID_OBSERVATION    |0           |49548 |
|INVALID_OBSERVATION    |2           |396159|
|INVALID_OBSERVATION    |7           |2     |
+-----------------------